# ELICIT robustness grid — Kaggle runner

One notebook runs **every cell of one served model**. A Kaggle session loads
one model, so the grid is partitioned by model alias and you run this notebook
once per alias per account.

## How to use it

1. Set `SERVED_MODEL` in the next cell to the alias you are about to serve.
2. Set `GRID` to the arm manifest you want (`configs/grid_arm_a.yaml`,
   `grid_arm_ab`, `grid_arm_d`, `grid_arm_c`, or `grid_full`).
3. Run all cells top to bottom. `elicit doctor` fails loudly rather than
   burning GPU-hours on a misconfigured session.
4. The archive at the end is the deliverable: download it, drop `results/`
   into the repo, then run the analysis pipeline.

## Why the grid is partitioned by model

The variance decomposition needs `sigma^2_model` to mean *model*. Holding the
serving configuration constant does that; running three models in one session
does not, because only one model is resident. `--served-model-name` must equal
the cell's declared alias, and `run_cell.py` refuses to start otherwise — a
quiet mismatch would silently relabel a whole arm's cells.

## Compute

Kaggle bills T4x2 as 2 GPU-hours per wall-clock hour, and a session includes
~30 min of setup (apt, `pip install vllm`, weight download, engine load). Pack
**two runs per session** to amortise it. With ToM off, one 26-agent 30-round run
is ~1.0-1.2 h wall-clock.

In [ ]:
# ============================================================================
# STEP 0 — configure this session
# ============================================================================
import json
import os
import subprocess
import sys
import time
from pathlib import Path

# Which served model this session will load. One of:
#   qwen2.5-14b | llama-3.1-8b | mistral-7b
SERVED_MODEL = "qwen2.5-14b"

# Which manifest to execute. Cells whose model is not SERVED_MODEL are skipped
# here, so this same notebook runs every arm.
GRID = "configs/grid_arm_a.yaml"

# Safety stop, so a mis-set GRID cannot spend a whole quota unnoticed.
MAX_CELLS = int(os.environ.get("ELICIT_MAX_CELLS", "0"))  # 0 = no limit

REPO = "https://github.com/uzairlol/ELICIT-fyp.git"
REPO_DIR = Path("/kaggle/working/elicit")

print(f"session serves : {SERVED_MODEL}")
print(f"grid manifest  : {GRID}")
print(f"max cells      : {MAX_CELLS or 'unlimited'}")
print(f"started        : {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")

!git log -1 --format='  HEAD: %h %s' 2>/dev/null || true

In [ ]:
# ============================================================================
# STEP 1 — clone, install Python dependencies
# ============================================================================
if not REPO_DIR.exists():
    !git clone -q {REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

print("Environment pre-check:")
!python --version
!free -h | head -2
!df -h /kaggle | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

!pip install -qq -r requirements.txt
!pip install -qq -e . --no-deps
print("dependencies installed.")

from core import parameters  # noqa: E402

print(f"ELICIT parameters importable; default backend = {parameters.LLM_BACKEND}")

In [ ]:
# ============================================================================
# STEP 2 — install vLLM into an isolated venv and verify CUDA
# ============================================================================
# Kaggle's base image ships RAPIDS, bigframes and protobuf 6.x, and a plain
# `pip install vllm` walks straight into broken dependency resolution there.
# An isolated venv keeps vLLM's resolver clean and leaves the base environment
# (which the simulation imports from) untouched; the two only talk over HTTP.
VENV = "/kaggle/working/vllm-env"

!apt-get update -y -qq > /dev/null && apt-get install -y -qq pciutils zstd > /dev/null

if not Path(VENV).exists():
    !python -m pip install -q --upgrade pip virtualenv
    !python -m virtualenv {VENV}

# Strip PYTHONPATH/PYTHONHOME: Kaggle's base sitecustomize imports 'wrapt',
# which the venv does not have, and it can interfere with vLLM's imports.
!env -u PYTHONPATH -u PYTHONHOME {VENV}/bin/pip install -q --upgrade pip
!env -u PYTHONPATH -u PYTHONHOME {VENV}/bin/pip install -q vllm huggingface_hub

print("vLLM environment:")
!env -u PYTHONPATH -u PYTHONHOME {VENV}/bin/python -c "import vllm, torch; print('  vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())"

In [ ]:
# ============================================================================
# STEP 3 — start the vLLM server for SERVED_MODEL
# ============================================================================
# The flags are held identical across families (tensor-parallel 2, max-model-len
# 8192, max-num-seqs 32) so the cross-model contrast stays a model contrast.
# Only --served-model-name and the checkpoint differ between sessions.
VLLM_LOG = "/kaggle/working/vllm.log"
HF_CACHE = "/kaggle/working/hf_cache"
os.environ["VLLM_BIN"] = f"{VENV}/bin/vllm"
os.environ["HF_HOME"] = HF_CACHE
os.makedirs(HF_CACHE, exist_ok=True)

serve_cmd = ["bash", "scripts/serve_arm_model.sh", SERVED_MODEL]
print("serve command:", " ".join(serve_cmd))

with open(VLLM_LOG, "w", encoding="utf-8") as handle:
    process = subprocess.Popen(
        serve_cmd,
        stdout=handle,
        stderr=subprocess.STDOUT,
        env={**os.environ, "VLLM_BIN": f"{VENV}/bin/vllm"},
    )
print(f"server pid {process.pid}; log -> {VLLM_LOG}")

# Weight download + engine load. ~30 min budget on a cold cache.
import urllib.error  # noqa: E402
import urllib.request  # noqa: E402

deadline = time.time() + 60 * 45
ready = False
while time.time() < deadline:
    if process.poll() is not None:
        print("server exited early; log tail:")
        !tail -40 {VLLM_LOG}
        raise SystemExit("vLLM failed to start")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5) as r:
            served = json.loads(r.read()).get("data", [])
        names = [m.get("id") for m in served]
        print("served models:", names)
        ready = SERVED_MODEL in names
        if ready:
            break
    except Exception:
        pass
    time.sleep(10)

if not ready:
    !tail -60 {VLLM_LOG}
    raise SystemExit(f"server did not come up as {SERVED_MODEL}")

print(f"=== server ready after {int(time.time() - (deadline - 60 * 45))}s ===")
!tail -6 {VLLM_LOG}

In [ ]:
# ============================================================================
# STEP 4 — smoke test: one real decision through the served engine
# ============================================================================
# Cheap insurance before a multi-hour grid: if the chat path or the fixed chat
# template is broken, find out in seconds rather than after run 1.
body = json.dumps(
    {
        "model": SERVED_MODEL,
        "messages": [{"role": "user", "content": "Reply with exactly: READY"}],
        "max_tokens": 8,
        "temperature": 0.0,
    }
).encode("utf-8")
request = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    data=body,
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(request, timeout=120) as response:
    reply = json.loads(response.read())
print("chat completion:", repr(reply["choices"][0]["message"]["content"].strip()))

sys.path.insert(0, str(REPO_DIR / "src"))
from llm.chat_template import render_chat  # noqa: E402

print("fixed chat template (used for grid parity):")
print(repr(render_chat("Reply with exactly: READY"))[:160])

raw = json.dumps(
    {
        "model": SERVED_MODEL,
        "prompt": render_chat("Reply with exactly: READY"),
        "max_tokens": 8,
        "temperature": 0.0,
    }
).encode("utf-8")
request = urllib.request.Request(
    "http://127.0.0.1:8000/v1/completions",
    data=raw,
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(request, timeout=120) as response:
    reply = json.loads(response.read())
print("raw completion:", repr(reply["choices"][0]["text"].strip()))

In [ ]:
# ============================================================================
# STEP 5 — enumerate this session's cells
# ============================================================================
# The notebook runs the grid through the CLI rather than reimplementing it, so
# what runs here is exactly what `elicit grid run` runs locally.
def elicit(*args):
    """Run the CLI in-process and return parsed JSON."""
    from cli import main as cli_main

    buffer = sys.stdout
    sys.stdout = (captured := __import__("io").StringIO())
    try:
        code = cli_main(list(args))
    finally:
        sys.stdout = buffer
    text = captured.getvalue()
    try:
        return code, json.loads(text[text.index("{") :])
    except ValueError:
        return code, {"raw": text}


code, plan = elicit("grid", "show", "--grid", GRID, "--arm", "A")
print(json.dumps({k: v for k, v in plan.items() if k != "cells_preview"}, indent=2))

all_cells = plan.get("cells_preview", [])
mine = [c for c in all_cells if c.get("model") == SERVED_MODEL]

if MAX_CELLS:
    mine = mine[:MAX_CELLS]

print(f"\nthis session will run {len(mine)} of {len(all_cells)} cells")
for cell in mine:
    print(
        f"  {cell['name']:<28} arm={cell['arm']} prompt={cell['prompt_variant']:<18} ",
        f"gov={cell['governance']} beta={cell['beta']}",
    )

if plan.get("parity_violations"):
    raise SystemExit(
        "PARITY VIOLATIONS — refusing to spend GPU-hours:\n"
        + json.dumps(plan["parity_violations"], indent=2)
    )

print(
    "\nparity check passed: no parity-critical parameter varies within a "
    "cross-model comparison group."
)

In [ ]:
# ============================================================================
# STEP 6 — preflight (`elicit doctor`)
# ============================================================================
# Run before every session. Each finding below corresponds to a way a run can
# silently become the wrong experiment:
#   governance_default_off            -> the G0 reference stops being the reference
#   chat_template_fixed_for_parity   -> P0 is a different string per family
#   seed_vector_complete             -> 'four seeds' may be one draw
#   shocks_decoupled_from_democracy  -> shock response == rule-change response
#   served_model_name_matches        -> the whole arm is relabelled
code, findings = elicit("doctor", "--model", SERVED_MODEL)
print(json.dumps(findings, indent=2))

if not findings.get("ok"):
    raise SystemExit("doctor reported a failing check; see above.")
print("\nall preflight checks passed.")

In [ ]:
# ============================================================================
# STEP 7 — execute this session's cells
# ============================================================================
# `--resume` skips cells already present in results/, so re-running this
# notebook after a quota cut costs nothing and duplicates nothing.
RUNTIME = int(os.environ.get("ELICIT_SESSION_BUDGET_S", str(7 * 3600)))

print(f"session runtime budget: {RUNTIME / 3600:.1f} h")
started = time.time()
completed = []
failed = []

for cell in mine:
    elapsed = time.time() - started
    if elapsed > RUNTIME:
        print(f"\n!! runtime budget reached after {completed} cells; stopping cleanly.")
        break

    print("\n" + "=" * 72)
    print(f"RUNNING {cell['name']}  (elapsed {elapsed / 3600:.2f} h)")
    print("=" * 72, flush=True)
    cell_started = time.time()
    try:
        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "cli",
                "run",
                "--preset",
                cell["name"].rsplit("_s", 1)[0],
                "--seed-index",
                cell["name"].rsplit("_s", 1)[1],
            ],
            cwd=str(REPO_DIR),
            env={**os.environ, "PYTHONPATH": str(REPO_DIR / "src")},
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr[-2000:])
        completed.append(cell["name"])
        print(f"DONE {cell['name']} in {(time.time() - cell_started) / 60:.1f} min")
    except Exception as exc:
        failed.append({"cell": cell["name"], "error": str(exc)[:1000]})
        print(f"FAILED {cell['name']}: {exc}")

print("\n" + "=" * 72)
print(
    json.dumps(
        {
            "served_model": SERVED_MODEL,
            "completed": completed,
            "failed": failed,
            "elapsed_hours": round((time.time() - started) / 3600, 2),
        },
        indent=2,
    )
)

In [ ]:
# ============================================================================
# STEP 8 — session report
# ============================================================================
# `elicit ls` reads only filenames, so it is instant even with many cells on
# disk. Headline numbers come from `elicit show` on the results themselves.
code, listing = elicit("ls", "--results-dir", "results")
print(f"cells on disk: {listing['count']}")
for row in listing["cells"]:
    print(f"  {row['arm']}  {row['cell']:<34} {row['bytes'] / 1e6:6.1f} MB")

print("\nresuming would skip these cells:")
!cd {REPO_DIR} && PYTHONPATH=src python -m cli ls --results-dir results | tail -3

In [ ]:
# ============================================================================
# STEP 9 — archive the deliverable
# ============================================================================
# results/ plus resolved_config.json is the whole handover. The resolved config
# is the artefact that makes the run re-instantiable: it carries the full seed
# vector, the model checkpoint and quantisation, the sampling settings, the
# prompt-variant hash and every effective parameter value.

archive = Path("/kaggle/working/elicit_results.tar.gz")
if archive.exists():
    archive.unlink()

!cd {REPO_DIR} && tar -czf {archive} results/

print("archive size:")
!ls -lh {archive}
print("\nresult files:")
!ls -1 {REPO_DIR}/results/simulation_*.json 2>/dev/null | wc -l
!ls -1 {REPO_DIR}/results/resolved_*.json 2>/dev/null | head -5

from IPython.display import FileLink  # noqa: E402

display(FileLink(str(archive)))

print("\n=== download the archive above, extract into the repo, then ===")
print("  python -m analysis.qwen_9seed.run_all --results-dir results")
print("=== to build the tables, and the new headline modules: ===")
print("  python -m analysis.revamp_run_all --results-dir results")